# 6. Model experiments

This notebook gathers the architecture and sensitivity tests into one place. Each experiment is a separate, clearly labeled test cell with a run switch. The network operations and single-target training implementation are defined here, so these tests no longer depend on a collection of experiment scripts.

These are legacy single-target NumPy model experiments. They use the existing train/development split; development scores guide checkpoint selection, so results are exploratory model-selection evidence, not independent test estimates. The historical test split is never loaded.

## 1. Packages and paths

In [ ]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import csv
import hashlib
import json
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'processed' / 'model_data_rates.npz').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'processed' / 'model_data_rates.npz').exists(): raise FileNotFoundError('Open this notebook inside the project.')
processedPath = rootPath / 'data' / 'processed'
resultsRoot = rootPath / 'results' / 'development'
modelsRoot = rootPath / 'models'

## 2. Network operations

In [ ]:
# Linear, ReLU, dropout, backpropagation, Adam, and prediction implementations.
def initialize_parameters_deep(layer_dims, seed=42, dtype=np.float32):
    """Create W1, b1, ..., WL, bL for dimensions such as [22, 32, 16, 1].

    Wl: (neurons in layer l, neurons in layer l-1).
    bl: (neurons in layer l, 1), broadcast across the examples.
    He initialization uses sqrt(2 / incoming neurons) for the ReLU network.
    """
    if(len(layer_dims) < 2 or layer_dims[-1] != 1):
        raise ValueError("Use input size, hidden sizes, and one regression output.")
    rng = np.random.default_rng(seed)
    parameters = {}

    L = len(layer_dims)

    for l in range(1, L):
        if(layer_dims[l - 1] < 1 or layer_dims[l] < 1):
            raise ValueError("Layer sizes must be positive.")
        scale = np.sqrt(2.0 / layer_dims[l - 1])
        parameters["W" + str(l)] = (rng.standard_normal((layer_dims[l], layer_dims[l - 1])) * scale).astype(dtype)
        parameters["b" + str(l)] = np.zeros((layer_dims[l], 1), dtype=dtype)

    return parameters

def linear_forward(A, W, b):
    """Calculate Z = W A + b and save the inputs needed for backpropagation."""
    Z = np.dot(W, A) + b
    cache = (A, W, b)
    return Z, cache

def relu(Z):
    """ReLU keeps positive values and replaces negative values with zero."""
    A = np.maximum(0, Z)
    cache = Z
    return A, cache

def relu_backward(dA, cache):
    """The derivative of ReLU is 1 for Z > 0 and 0 for Z <= 0."""
    Z = cache
    dZ = dA * (Z > 0)
    return dZ

def linear_activation_forward(A_prev, W, b, activation):
    """One LINEAR -> RELU or LINEAR -> LINEAR layer, as in the notebook."""
    Z, linear_cache = linear_forward(A_prev, W, b)

    if(activation == "relu"):
        A, activation_cache = relu(Z)
    elif(activation == "linear"):
        A = Z
        activation_cache = Z
    else:
        raise ValueError("Activation must be relu or linear.")

    cache = (linear_cache, activation_cache)
    return A, cache

def L_model_forward(X, parameters, keep_prob=1.0, training=False, rng=None):
    """Run [LINEAR -> RELU -> optional DROPOUT] * (L-1) -> LINEAR.

    keep_prob is the probability of KEEPING an activation: 0.9 means 10%
    dropout. Each cache contains (layer_cache, D), where D is that hidden
    layer's dropout mask, or None. Prediction always disables dropout.
    Pass a persistent random generator when training with dropout; resetting
    the seed inside this function would repeat the same masks every batch.
    """
    if(not np.isfinite(keep_prob) or keep_prob <= 0 or keep_prob > 1):
        raise ValueError("keep_prob must be greater than zero and at most one.")
    if(training and keep_prob < 1 and rng is None):
        raise ValueError("Supply a random generator for training with dropout.")

    A = X
    caches = []
    L = len(parameters) // 2

    for l in range(1, L):
        A_prev = A
        A, layer_cache = linear_activation_forward(A_prev, parameters["W" + str(l)], parameters["b" + str(l)], "relu")
        D = None

        if(training and keep_prob < 1):
            # The same four dropout steps as Regularization.ipynb.
            D = rng.random(A.shape)
            D = (D < keep_prob).astype(A.dtype)
            A = A * D
            A = A / keep_prob

        caches.append((layer_cache, D))

    AL, layer_cache = linear_activation_forward(A, parameters["W" + str(L)], parameters["b" + str(L)], "linear")
    caches.append((layer_cache, None))
    return AL, caches

def forward_propagation_with_dropout(X, parameters, keep_prob=0.5, rng=None):
    """Notebook-style training entry point; dropout applies to hidden layers."""
    return L_model_forward(X, parameters, keep_prob, training=True, rng=rng)

def compute_cost(AL, Y, parameters=None, weight_decay=0.0):
    """MSE = sum((AL - Y)^2) / m, plus optional coupled L2.

    Coupled L2 applies to weights AND biases, matching the existing experiment.
    It is not divided by m; weight_decay is the coefficient of this penalty.
    """
    if(AL.shape != Y.shape or AL.ndim != 2 or AL.shape[0] != 1):
        raise ValueError("Predictions and targets must both have shape (1, examples).")
    if(weight_decay != 0 and parameters is None):
        raise ValueError("Parameters are required to calculate the L2 penalty.")

    m = Y.shape[1]
    cost = float(np.sum((AL - Y) ** 2) / m)

    if(parameters is not None):
        for value in parameters.values():
            cost = cost + 0.5 * weight_decay * float(np.sum(value ** 2))

    return cost

def linear_backward(dZ, cache):
    """Backpropagate through Z = W A_prev + b.

    As in the notebook, average dW and db over the m examples here. dA_prev
    remains unaveraged so the preceding layer does not average it twice.
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]
    dW = (1.0 / m) * np.dot(dZ, A_prev.T)
    db = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)
    return dA_prev, dW, db

def linear_activation_backward(dA, cache, activation):
    """Apply the activation derivative, then backpropagate the linear part."""
    linear_cache, activation_cache = cache

    if(activation == "relu"):
        dZ = relu_backward(dA, activation_cache)
    elif(activation == "linear"):
        dZ = dA
    else:
        raise ValueError("Activation must be relu or linear.")

    dA_prev, dW, db = linear_backward(dZ, linear_cache)
    return dA_prev, dW, db

def L_model_backward(AL, Y, caches, keep_prob=1.0, weight_decay=0.0):
    """Return notebook-style grads: dW1, db1, ..., dWL, dbL, plus dA values.

    MSE starts with 2 * (AL - Y). linear_backward supplies the factor 1/m.
    Use the SAME keep_prob and cached masks as in the forward pass.
    Stored dA values use the notebook's unaveraged-per-example convention.
    """
    if(AL.shape != Y.shape or AL.ndim != 2 or AL.shape[0] != 1):
        raise ValueError("Predictions and targets must both have shape (1, examples).")
    if(not np.isfinite(keep_prob) or keep_prob <= 0 or keep_prob > 1):
        raise ValueError("keep_prob must be greater than zero and at most one.")

    grads = {}
    L = len(caches)
    dAL = 2.0 * (AL - Y)

    layer_cache, D = caches[L - 1]
    dA_prev, dW, db = linear_activation_backward(dAL, layer_cache, "linear")
    linear_cache, activation_cache = layer_cache
    A_prev, W, b = linear_cache
    grads["dW" + str(L)] = dW + weight_decay * W
    grads["db" + str(L)] = db + weight_decay * b
    grads["dA" + str(L - 1)] = dA_prev

    for l in reversed(range(1, L)):
        layer_cache, D = caches[l - 1]
        dA = grads["dA" + str(l)]

        if(D is not None):
            # Reuse the forward mask and apply the same 1/keep_prob scaling.
            dA = dA * D
            dA = dA / keep_prob

        dA_prev, dW, db = linear_activation_backward(dA, layer_cache, "relu")
        linear_cache, activation_cache = layer_cache
        A_prev, W, b = linear_cache
        grads["dW" + str(l)] = dW + weight_decay * W
        grads["db" + str(l)] = db + weight_decay * b
        grads["dA" + str(l - 1)] = dA_prev

    return grads

def backward_propagation_with_dropout(AL, Y, caches, keep_prob, weight_decay=0.0):
    """Notebook-style dropout backward pass, generalized to any hidden sizes."""
    return L_model_backward(AL, Y, caches, keep_prob, weight_decay)

def initialize_adam(parameters):
    """v stores moving-average gradients; s stores moving-average squares."""
    v = {}
    s = {}

    for key, value in parameters.items():
        v["d" + key] = np.zeros_like(value)
        s["d" + key] = np.zeros_like(value)

    return v, s

def update_parameters_with_adam(parameters, grads, v, s, t, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
    """Update parameters in place, returning the five values from the notebook.

    t counts MINI-BATCH UPDATES, not epochs. Bias correction depends on t.
    """
    v_corrected = {}
    s_corrected = {}

    for key in parameters:
        gradient_key = "d" + key
        v[gradient_key] = beta1 * v[gradient_key] + (1.0 - beta1) * grads[gradient_key]
        s[gradient_key] = beta2 * s[gradient_key] + (1.0 - beta2) * grads[gradient_key] ** 2
        v_corrected[gradient_key] = v[gradient_key] / (1.0 - beta1 ** t)
        s_corrected[gradient_key] = s[gradient_key] / (1.0 - beta2 ** t)
        parameters[key] -= learning_rate * v_corrected[gradient_key] / (np.sqrt(s_corrected[gradient_key]) + epsilon)

    return parameters, v, s, v_corrected, s_corrected

def update_parameters(parameters, grads, learning_rate):
    """The simpler gradient-descent update from the step-by-step notebook."""
    for key in parameters:
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

def copy_parameters(parameters):
    copied = {}

    for key, value in parameters.items():
        copied[key] = value.copy()

    return copied

def predict(inputs, parameters, target_mean=0.0, target_scale=1.0, prediction_minimum=None):
    predictions, caches = L_model_forward(inputs, parameters)
    predictions = predictions.reshape(-1) * target_scale + target_mean

    if(prediction_minimum is not None):
        predictions = np.maximum(predictions, prediction_minimum)

    return predictions

def load_model(path):
    """Return parameters and metadata; inference inputs have shape (features, examples)."""
    import json
    parameters = {}

    with np.load(path, allow_pickle=False) as checkpoint:
        metadata = json.loads(str(checkpoint["metadata"].item()))

        for layer in range(1, len(metadata["layer_dims"])):
            for prefix in ["W", "b"]:
                key = prefix + str(layer)
                parameters[key] = checkpoint[key].copy()

    return parameters, metadata

## 3. Training implementation

In [ ]:
def metrics(predictions, targets):
    errors = predictions.astype(np.float64) - targets.astype(np.float64)
    return float(np.sqrt(np.mean(errors ** 2))), float(np.mean(np.abs(errors)))

def loadTargetArrays(data, target, split):
    """Minutes are an observed NEXT-season target, never a future input feature."""
    if(target == "minutes"):
        actual = data["next_season_minutes_" + split]
        baseline = data["prior_minutes_" + split]

        if(not np.isfinite(actual).all() or not np.isfinite(baseline).all() or (actual < 0).any() or (baseline < 0).any()):
            raise ValueError("Minutes and prior-minute baselines must be finite and nonnegative.")
    else:
        actual = data["y_" + target + "_" + split]
        baseline = data["baseline_" + target + "_" + split]

    return actual.copy(), baseline.copy()

def trainTarget(target, args):
    filename = "model_data.npz"

    if(args.dataset == "rates"):
        filename = "model_data_rates.npz"

    data_path = args.data_dir / filename
    output = args.output / args.dataset / target
    output.mkdir(parents=True, exist_ok=True)

    # Never access test arrays during training or model selection.
    with np.load(data_path, allow_pickle=False) as data:
        x_train = data["X_train"].astype(np.float32)
        x_dev = data["X_dev"].astype(np.float32)
        y_train, unused = loadTargetArrays(data, target, "train")
        y_dev, baseline = loadTargetArrays(data, target, "dev")
        y_train = y_train.astype(np.float32)
        y_dev = y_dev.astype(np.float32)
        baseline = baseline.astype(np.float64)
        feature_names = data["feature_names"].tolist()

        # PIE retains its original training rows. Rate targets exclude undefined rates.
        if(target in ["ws_48", "bpm"]):
            train_mask = data["rate_mask_train"].astype(bool)
            dev_mask = data["rate_mask_dev"].astype(bool)
            x_train = x_train[train_mask]
            y_train = y_train[train_mask]
            x_dev = x_dev[dev_mask]
            y_dev = y_dev[dev_mask]
            baseline = baseline[dev_mask]

    if(x_train.ndim != 2 or x_dev.ndim != 2 or x_train.shape[1] != x_dev.shape[1]):
        raise ValueError("Training and development inputs must have matching feature counts.")
    if(y_train.shape != (len(x_train),) or y_dev.shape != (len(x_dev),) or baseline.shape != y_dev.shape):
        raise ValueError("Unexpected target or baseline shapes.")
    if(len(x_train) == 0 or len(x_dev) == 0 or len(feature_names) != x_train.shape[1]):
        raise ValueError("Empty split or inconsistent feature names.")

    for values in [x_train, x_dev, y_train, y_dev, baseline]:
        if(not np.isfinite(values).all()):
            raise ValueError("Inputs, targets, and baseline must be finite.")

    # Preserve the previous training convention: PIE unscaled, other targets standardized.
    target_mean = 0.0
    target_scale = 1.0
    prediction_minimum = None

    if(target == "minutes"):
        # The linear network learns standardized minutes with MSE. At inference,
        # floor the decoded minute prediction at zero, including dev selection.
        prediction_minimum = 0.0

    if(target != "pie"):
        target_mean = float(np.mean(y_train))
        target_scale = float(np.std(y_train))

        if(target_scale <= 0):
            raise ValueError("Training target has no variation.")

    training_targets = ((y_train - target_mean) / target_scale).reshape(1, -1)
    x_train = x_train.T
    x_dev = x_dev.T
    layer_dims = [x_train.shape[0]] + args.hidden + [1]
    baseline_rmse, baseline_mae = metrics(baseline, y_dev)
    best_overall = float("inf")
    history = []
    runs = []
    started = time.perf_counter()
    print ("\nTraining target: " + target.upper() + " | layer dimensions: " + str(layer_dims), flush=True)
    print ("Each column is one example. Hidden layers use ReLU; the output is linear.")

    for l in range(1, len(layer_dims)):
        print ("Layer " + str(l) + ": W" + str(l) + " shape " + str((layer_dims[l], layer_dims[l - 1])) + " | b" + str(l) + " shape " + str((layer_dims[l], 1)))

    for dropout in args.dropout:
        # CLI dropout is a DROP probability; the notebook uses a KEEP probability.
        keep_prob = 1.0 - dropout
        parameters = initialize_parameters_deep(layer_dims, args.seed)
        v, s = initialize_adam(parameters)
        shuffle_rng = np.random.default_rng(args.seed + 1)
        dropout_rng = np.random.default_rng(args.seed + 2)
        best_rmse = float("inf")
        stale_epochs = 0
        step = 0

        for epoch in range(1, args.epochs + 1):
            order = shuffle_rng.permutation(x_train.shape[1])
            squared_error = 0.0

            for start in range(0, len(order), args.batch_size):
                indices = order[start:start + args.batch_size]
                X = x_train[:, indices]
                Y = training_targets[:, indices]

                # 1. Forward propagation: inputs -> hidden activations -> prediction.
                AL, caches = L_model_forward(X, parameters, keep_prob, True, dropout_rng)

                # 2. Backpropagation: prediction errors -> dW and db for each layer.
                grads = L_model_backward(AL, Y, caches, keep_prob, args.weight_decay)

                # 3. Adam uses those gradients to update every W and b.
                step = step + 1
                parameters, v, s, v_corrected, s_corrected = update_parameters_with_adam(parameters, grads, v, s, step, args.learning_rate)
                squared_error = squared_error + float(np.sum((AL - Y) ** 2))

            train_mse = squared_error / x_train.shape[1]
            # Compare both splits at the SAME end-of-epoch weights, dropout off.
            train_predictions = predict(x_train, parameters, target_mean, target_scale, prediction_minimum)
            train_rmse, train_mae = metrics(train_predictions, y_train)
            dev_predictions = predict(x_dev, parameters, target_mean, target_scale, prediction_minimum)
            dev_rmse, dev_mae = metrics(dev_predictions, y_dev)

            if(not np.isfinite(train_mse) or not np.isfinite(train_rmse) or not np.isfinite(dev_rmse)):
                raise ValueError("Training produced a nonfinite loss.")

            history.append({"dropout": dropout, "epoch": epoch, "train_mse_scaled": train_mse, "train_eval_mse_scaled": (train_rmse / target_scale) ** 2, "train_mse": train_rmse ** 2, "train_rmse": train_rmse, "train_mae": train_mae, "dev_mse": dev_rmse ** 2, "dev_rmse": dev_rmse, "dev_mae": dev_mae})

            if(dev_rmse < best_rmse):
                best_rmse = dev_rmse
                best_epoch = epoch
                best_state = copy_parameters(parameters)
                stale_epochs = 0
            else:
                stale_epochs = stale_epochs + 1

            if(epoch == 1 or epoch % 25 == 0):
                print ("Dropout " + str(dropout) + " | epoch " + str(epoch) + " | train RMSE " + str(round(train_rmse, 6)) + " | dev RMSE " + str(round(dev_rmse, 6)), flush=True)

            if(stale_epochs >= args.patience):
                break

        predictions = predict(x_dev, best_state, target_mean, target_scale, prediction_minimum)
        dev_rmse, dev_mae = metrics(predictions, y_dev)
        train_predictions = predict(x_train, best_state, target_mean, target_scale, prediction_minimum)
        train_rmse, train_mae = metrics(train_predictions, y_train)
        result = {"dropout": dropout, "best_epoch": best_epoch, "epochs_run": epoch, "train_mse": train_rmse ** 2, "train_eval_mse_scaled": (train_rmse / target_scale) ** 2, "train_rmse": train_rmse, "train_mae": train_mae, "dev_mse": dev_rmse ** 2, "dev_rmse": dev_rmse, "dev_mae": dev_mae, "reached_epoch_limit": epoch == args.epochs}
        runs.append(result)

        if(dev_rmse < best_overall):
            best_overall = dev_rmse
            selected = result.copy()
            selected_state = copy_parameters(best_state)
            selected_predictions = predictions.copy()

    metadata = {"target": target, "dataset": args.dataset, "layer_dims": layer_dims, "feature_names": feature_names, "target_mean": target_mean, "target_scale": target_scale, "selected": selected, "runs": runs, "baseline_dev_rmse": baseline_rmse, "baseline_dev_mae": baseline_mae, "learning_rate": args.learning_rate, "weight_decay": args.weight_decay, "regularization": "coupled L2 on weights and biases", "batch_size": args.batch_size, "patience": args.patience, "maximum_epochs": args.epochs, "seed": args.seed, "training_examples": x_train.shape[1], "development_examples": len(y_dev), "model_data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(), "test_evaluated": False, "elapsed_seconds": time.perf_counter() - started}
    metadata["prediction_minimum"] = prediction_minimum

    if(target == "minutes"):
        metadata["target_definition"] = "Observed next-season regular-season total minutes; conditional on appearing in the following season."
        metadata["prediction_policy"] = "Linear standardized-target MSE training; decoded predictions floored at zero for inference and development selection."
    checkpoint = copy_parameters(selected_state)
    checkpoint["metadata"] = np.array(json.dumps(metadata, allow_nan=False))
    np.savez(output / "best_model.npz", **checkpoint)
    np.savez(output / "development_predictions.npz", predicted=selected_predictions, actual=y_dev, baseline=baseline)
    (output / "development_results.json").write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8")

    with (output / "training_history.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

    print ("Selected dropout: " + str(selected["dropout"]))
    print ("Best epoch: " + str(selected["best_epoch"]))
    print ("Selected checkpoint training MSE: " + str(selected["train_mse"]))
    print ("Selected checkpoint development MSE: " + str(selected["dev_mse"]))
    print ("Development model RMSE: " + str(selected["dev_rmse"]))
    print ("Development baseline RMSE: " + str(baseline_rmse))
    print ("Development model MAE: " + str(selected["dev_mae"]))
    print ("Saved model to: " + str(output / "best_model.npz"))

### Shared experiment runner

In [ ]:
experimentSettings = {'batchSize':64,'epochs':300,'patience':25,'batchSize':64,'weightDecay':0.0001}

def runGrid(experimentName, testCases, seeds, dropoutOptions):
    runId = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    modelFolder = modelsRoot / (experimentName + '_experiments') / runId
    resultFolder = resultsRoot / (experimentName + '_experiments') / runId
    resultFolder.mkdir(parents=True, exist_ok=False)
    rows = []
    for testCase in testCases:
        for seed in seeds:
            target = testCase['target']
            outputPath = modelFolder / testCase['name'] / ('seed_' + str(seed))
            args = SimpleNamespace(
                dataset='rates', data_dir=processedPath, output=outputPath,
                hidden=testCase['hidden'], dropout=dropoutOptions,
                epochs=experimentSettings['epochs'], patience=experimentSettings['patience'],
                batch_size=experimentSettings['batchSize'],
                learning_rate=testCase.get('learningRate',0.001),
                weight_decay=experimentSettings['weightDecay'], seed=seed,
            )
            print('Test: ' + experimentName + ' | ' + target.upper() + ' | ' + testCase['name'] + ' | seed ' + str(seed), flush=True)
            trainTarget(target, args)
            modelPath = outputPath / 'rates' / target
            report = json.loads((modelPath / 'development_results.json').read_text(encoding='utf-8'))
            for run in report['runs']:
                rows.append({
                    'target':target,'candidate':testCase['name'],
                    'hiddenWidths':'-'.join(map(str,testCase['hidden'])),'seed':seed,
                    'learningRate':args.learning_rate,'dropout':run['dropout'],
                    'devRmse':run['dev_rmse'],'devMae':run['dev_mae'],
                    'baselineDevRmse':report['baseline_dev_rmse'],
                    'bestEpoch':run['best_epoch'],'testEvaluated':False,
                })
    resultTable = pd.DataFrame(rows)
    summaryTable = resultTable.groupby(['target','candidate','hiddenWidths','dropout'],as_index=False).agg(
        seeds=('seed','nunique'),devRmseMean=('devRmse','mean'),devRmseSd=('devRmse','std'),devMaeMean=('devMae','mean'))
    resultTable.to_csv(resultFolder / 'experiment_results.csv',index=False)
    summaryTable.to_csv(resultFolder / 'experiment_summary.csv',index=False)
    print('Saved: ' + str(resultFolder))
    return resultTable, summaryTable, resultFolder

## 4. Test A — hidden-layer depth

In [ ]:
depthRates = {'pie':0.001,'ws_48':0.001,'bpm':0.003,'minutes':0.001}
depthTests = [{'target':target,'name':'depth_' + str(depth),'hidden':[32] * depth,'learningRate':depthRates[target]} for target in ['pie','ws_48','bpm','minutes'] for depth in range(1,11)]
runDepthTest = False
if runDepthTest: depthResults, depthSummary, depthFolder = runGrid('depth',depthTests,[42],[0.1])
print('Depth cases: ' + str(len(depthTests)) + ' | enabled: ' + str(runDepthTest))

## 5. Test B — target-specific width

In [ ]:
widthTests = []
for target,widths in {'pie':[[16,16,16],[32,32,32],[64,64,64]],'ws_48':[[16],[32],[64],[128]],'bpm':[[16],[32],[64],[128]],'minutes':[[16],[32],[64],[128]]}.items():
    rate = 0.003 if target == 'bpm' else 0.001
    for hidden in widths: widthTests.append({'target':target,'name':target + '_' + 'x'.join(map(str,hidden)),'hidden':hidden,'learningRate':rate})
runWidthTest = False
if runWidthTest: widthResults, widthSummary, widthFolder = runGrid('width',widthTests,[42],[0.1])
print('Width cases: ' + str(len(widthTests)) + ' | enabled: ' + str(runWidthTest))

## 6. Test C — architecture stability across seeds

In [ ]:
seedTests = [
    {'target':'pie','name':'current_32_16','hidden':[32,16]}, {'target':'pie','name':'single_32','hidden':[32]}, {'target':'pie','name':'three_layer_32','hidden':[32,32,32]},
    {'target':'ws_48','name':'current_128_64','hidden':[128,64]}, {'target':'ws_48','name':'single_16','hidden':[16]}, {'target':'ws_48','name':'single_32','hidden':[32]},
    {'target':'bpm','name':'current_32_16','hidden':[32,16],'learningRate':0.003}, {'target':'bpm','name':'single_16','hidden':[16],'learningRate':0.003}, {'target':'bpm','name':'single_64','hidden':[64],'learningRate':0.003},
    {'target':'minutes','name':'current_64_32','hidden':[64,32]}, {'target':'minutes','name':'single_32','hidden':[32]}, {'target':'minutes','name':'single_64','hidden':[64]},
]
runSeedTest = False
if runSeedTest: seedResults, seedSummary, seedFolder = runGrid('architecture_multiseed',seedTests,[42,43,44,45,46],[0.1])
print('Seed cases: ' + str(len(seedTests) * 5) + ' | enabled: ' + str(runSeedTest))

## 7. Test D — regularization and learning rate

In [ ]:
regularizationTests = [
    {'target':target,'name':name,'hidden':hidden,'learningRate':rate}
    for target,rate in [('pie',0.001),('ws_48',0.001),('bpm',0.003)]
    for name,hidden in [('tiny_4_2',[4,2]),('small_16_8',[16,8]),('reference_32_16',[32,16]),('wide_64_32',[64,32]),('slow_32_16',[32,16]),('fast_32_16',[32,16])]
]
for testCase in regularizationTests:
    if testCase['name'] == 'slow_32_16': testCase['learningRate'] = 0.0003
    if testCase['name'] == 'fast_32_16': testCase['learningRate'] = 0.003
runRegularizationTest = False
if runRegularizationTest: regularizationResults, regularizationSummary, regularizationFolder = runGrid('regularization',regularizationTests,[42],[0.0,0.1,0.2,0.3,0.5])
print('Regularization cases: ' + str(len(regularizationTests)) + ' | enabled: ' + str(runRegularizationTest))

## 8. Test E — two-hidden-layer widths across seeds

In [ ]:
twoLayerTests = []
twoLayerWidths = {
    'pie':{'narrow_16_8':[16,8],'current_32_16':[32,16],'wide_64_32':[64,32],'large_128_64':[128,64]},
    'ws_48':{'narrow_16_8':[16,8],'medium_32_16':[32,16],'wide_64_32':[64,32],'current_128_64':[128,64]},
    'bpm':{'narrow_16_8':[16,8],'current_32_16':[32,16],'wide_64_32':[64,32],'large_128_64':[128,64]},
    'minutes':{'narrow_16_8':[16,8],'medium_32_16':[32,16],'current_64_32':[64,32],'large_128_64':[128,64]},
}
for target,widths in twoLayerWidths.items():
    rate = 0.003 if target == 'bpm' else 0.001
    for candidate,hidden in widths.items(): twoLayerTests.append({'target':target,'name':candidate,'hidden':hidden,'learningRate':rate})
runTwoLayerTest = False
if runTwoLayerTest: twoLayerResults, twoLayerSummary, twoLayerFolder = runGrid('two_layer_width_multiseed',twoLayerTests,[42,43,44,45,46],[0.1])
print('Two-layer seed runs: ' + str(len(twoLayerTests) * 5) + ' | enabled: ' + str(runTwoLayerTest))

## 9. Test F — three-hidden-layer widths across seeds

In [ ]:
threeLayerWidths = {
    'narrow_16_8_4':[16,8,4],
    'medium_32_16_8':[32,16,8],
    'equal_32_32_32':[32,32,32],
    'wide_64_32_16':[64,32,16],
}
threeLayerRates = {'pie':0.001,'ws_48':0.001,'bpm':0.003,'minutes':0.001}
threeLayerTests = [
    {'target':target,'name':candidate,'hidden':hidden,'learningRate':threeLayerRates[target]}
    for target in ['pie','ws_48','bpm','minutes']
    for candidate,hidden in threeLayerWidths.items()
]
runThreeLayerTest = False
if runThreeLayerTest: threeLayerResults, threeLayerSummary, threeLayerFolder = runGrid('three_layer_width_multiseed',threeLayerTests,[42,43,44,45,46],[0.1])
print('Three-layer seed runs: ' + str(len(threeLayerTests) * 5) + ' | enabled: ' + str(runThreeLayerTest))

## 10. Review saved experiment results

In [ ]:
savedExperiments = {
    'Depth':('numpy_depth_experiments','depth_comparison.csv'),
    'Width':('numpy_width_experiments','width_comparison.csv'),
    'Architecture across seeds':('numpy_multiseed_experiments','architecture_summary.csv'),
    'Two-layer widths across seeds':('numpy_two_layer_width_experiments','width_summary.csv'),
    'Three-layer widths across seeds':('numpy_three_layer_width_experiments','width_summary.csv'),
}
historicalTables = {}
for label,(folderName,fileName) in savedExperiments.items():
    candidates = sorted((resultsRoot / folderName).glob('*/' + fileName))
    if candidates:
        print(label + ' | ' + candidates[-1].parent.name)
        historicalTables[label] = pd.read_csv(candidates[-1])
        display(historicalTables[label].head(12))

sensitivityCandidates = sorted((resultsRoot / 'multitask_profile').glob('*/minimum_games_sensitivity_comparison.csv'))
if sensitivityCandidates:
    sensitivityPath = sensitivityCandidates[-1]
    sensitivityResults = pd.read_csv(sensitivityPath)
    display(sensitivityResults[sensitivityResults.target.isin(['pie','ws_48','bpm','minutes'])].round(4))
else:
    print('No saved shared-profile low-game sensitivity result found.')

## 11. Visualize experiment comparisons inline

These panels read the saved train/development summaries and render directly in the notebook. Error bars show between-seed standard deviation where available.


In [ ]:
import matplotlib.pyplot as plt

plotSpecs = [
    ('Depth','depth','dev_rmse'),
    ('Width','hidden_widths','dev_rmse'),
    ('Architecture across seeds','candidate','dev_rmse_mean'),
    ('Two-layer widths across seeds','candidate','dev_rmse_mean'),
    ('Three-layer widths across seeds','configuration','dev_rmse_mean'),
]
availableSpecs = [(label,xColumn,yColumn) for label,xColumn,yColumn in plotSpecs if label in historicalTables]
plotRows = max(1, (len(availableSpecs) + 1) // 2)
experimentFigure, experimentAxes = plt.subplots(plotRows, 2, figsize=(14, 5 * plotRows), squeeze=False)

for axis, (label,xColumn,yColumn) in zip(experimentAxes.flat, availableSpecs):
    table = historicalTables[label]
    for targetName, targetRows in table.groupby('target'):
        targetRows = targetRows.sort_values(xColumn)
        xValues = targetRows[xColumn].astype(str).to_numpy()
        yValues = targetRows[yColumn].to_numpy(dtype=float)
        errorValues = targetRows['dev_rmse_sd'].to_numpy(dtype=float) if 'dev_rmse_sd' in targetRows else None
        axis.errorbar(xValues, yValues, yerr=errorValues, marker='o', capsize=3, label=targetName)
    axis.set_title(label)
    axis.set_xlabel(xColumn.replace('_',' '))
    axis.set_ylabel('Development RMSE (lower is better)')
    axis.tick_params(axis='x', rotation=35)
    axis.legend()

for axis in experimentAxes.flat[len(availableSpecs):]: axis.set_visible(False)
experimentFigure.tight_layout()
experimentPlotPath = resultsRoot / 'model_experiments_summary.png'
experimentFigure.savefig(experimentPlotPath, dpi=160)
display(experimentFigure)
print('Saved summary plot: ' + str(experimentPlotPath))

## 12. Interpretation notes

- Only enable one experiment switch at a time; a full seed/depth sweep can take a long time.
- Results are selected and compared on development data, so do not report them as independent test results.
- The legacy one-target networks differ from the active shared profile network.
- The current model's 2026 salary holdout is handled separately in notebook 05.